# Phase 1 & 1.5: Multi-Horizon & Contextual Data Collection

This notebook serves as the foundation for our Hybrid Temporal Forecaster. We are building a dual-horizon dataset:
1. **Strategic Daily Data (20 Years)**: To capture long-term market regimes and economic memory.
2. **Tactical Hourly Data (2 Years)**: To capture intra-day volatility and short-term forecasting patterns.
3. **Contextual Market Indicators (Phase 1.5)**: Adding VIX, Bond Yields, and Sector Proxies to enrich the feature space.

## 1. Imports and Reproducibility

As always, we establish a global seed of `25` to ensure that any stochastic data operations remain perfectly reproducible.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

SEED = 25
np.random.seed(SEED)
random.seed(SEED)

print(f"Global seed established: {SEED}")

Global seed established: 25


## 2. Configuration and Horizons

We target a universe of core assets (AAPL, MSFT, JPM, SPY, GLD) and augment them with macro indicators (^VIX, ^TNX, DX-Y.NYB) and sector ETFs (XLK, XLF).

In [2]:
CORE_TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD"]
MACRO_TICKERS = ["^VIX", "^TNX", "DX-Y.NYB", "XLK", "XLF"]
ALL_TICKERS = CORE_TICKERS + MACRO_TICKERS

END_DATE = datetime.now().strftime("%Y-%m-%d")

# Daily configuration (~20 years)
DAILY_START_DATE = (datetime.now() - timedelta(days=365*20)).strftime("%Y-%m-%d")
DAILY_DIR = "../data/raw/daily"

# Hourly configuration (~2 years)
HOURLY_START_DATE = (datetime.now() - timedelta(days=729)).strftime("%Y-%m-%d")
HOURLY_DIR = "../data/raw/hourly"

os.makedirs(DAILY_DIR, exist_ok=True)
os.makedirs(HOURLY_DIR, exist_ok=True)

print(f"Daily horizon: {DAILY_START_DATE} to {END_DATE}")
print(f"Hourly horizon: {HOURLY_START_DATE} to {END_DATE}")

Daily horizon: 2006-03-27 to 2026-03-22
Hourly horizon: 2024-03-23 to 2026-03-22


## 3. Implementation of the Ingestion Pipeline

We define a unified downloader that handles different intervals (1d, 1h) and stores them in organized subdirectories.

In [3]:
def fetch_data(tickers, start, end, interval, target_dir):
    for ticker in tickers:
        print(f"Processing {ticker} at {interval} interval...")
        try:
            df = yf.download(ticker, start=start, end=end, interval=interval)
            if not df.empty:
                # Clean ticker name for filename (remove symbols like ^)
                clean_name = ticker.replace("^", "").replace("-Y.NYB", "").replace(".", "_")
                path = os.path.join(target_dir, f"{clean_name}.csv")
                df.to_csv(path)
                print(f"  -> Saved to {path} ({len(df)} rows)")
            else:
                print(f"  !! Warning: Failed to fetch {ticker} (Empty DF)")
        except Exception as e:
            print(f"  !! Error fetching {ticker}: {e}")

print("Starting Strategic Data Collection (Daily)...")
fetch_data(ALL_TICKERS, DAILY_START_DATE, END_DATE, "1d", DAILY_DIR)

print("\nStarting Tactical Data Collection (Hourly)...")
fetch_data(ALL_TICKERS, HOURLY_START_DATE, END_DATE, "1h", HOURLY_DIR)

Starting Strategic Data Collection (Daily)...
Processing AAPL at 1d interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/daily/AAPL.csv (5028 rows)
Processing MSFT at 1d interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/daily/MSFT.csv (5028 rows)
Processing JPM at 1d interval...


...

  -> Saved to ../data/raw/hourly/DX.csv (11876 rows)
Processing XLK at 1h interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/hourly/XLK.csv (3459 rows)
Processing XLF at 1h interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/hourly/XLF.csv (3459 rows)


## 4. Final Verification and Storage Audit

We conclude with a quick cross-reference to ensure that all assets across both horizons are properly cached.

In [4]:
audit = []
CLEAN_NAMES = [t.replace("^", "").replace("-Y.NYB", "").replace(".", "_") for t in ALL_TICKERS]

for horizon, path in [("Daily", DAILY_DIR), ("Hourly", HOURLY_DIR)]:
    for name in CLEAN_NAMES:
        file_path = os.path.join(path, f"{name}.csv")
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            audit.append({
                "Horizon": horizon,
                "Ticker": name,
                "Rows": len(df),
                "Start": df.iloc[0, 0],
                "End": df.iloc[-1, 0]
            })

pd.DataFrame(audit)

,Horizon,Ticker,Rows,Start,End
0,Daily,AAPL,5030,Ticker,2026-03-20
1,Daily,MSFT,5030,Ticker,2026-03-20
2,Daily,JPM,5030,Ticker,2026-03-20
3,Daily,SPY,5030,Ticker,2026-03-20
4,Daily,GLD,5030,Ticker,2026-03-20
5,Daily,VIX,5030,Ticker,2026-03-20
6,Daily,TNX,5027,Ticker,2026-03-20
7,Daily,DX,5033,Ticker,2026-03-20
8,Daily,XLK,5030,Ticker,2026-03-20
9,Daily,XLF,5030,Ticker,2026-03-20
